# 01 Data Audit

**Objective:** Perform comprehensive exploratory data analysis (EDA) and data quality checks on the Tashkent house prices dataset.

---

## 📋 Data Audit Checklist

- [ ] **Dataset Shape & Size**: rows, columns, memory usage
- [ ] **Column Names & Data Types**: verify correct dtypes; identify categorical vs. numeric
- [ ] **Missing Values**: absolute count, percentage per column
- [ ] **Duplicate Rows**: identify and quantify exact duplicates
- [ ] **Target Variable (Price)**: distribution, outliers, missing values
- [ ] **Feature Distributions**: univariate stats (mean, median, std, min, max)
- [ ] **Categorical Features**: unique values, value counts for top categories
- [ ] **Numerical Features**: range, skewness, kurtosis, outliers (IQR method)
- [ ] **Data Leakage Risks**: identify redundant or derived features that might leak target info
- [ ] **Outliers & Anomalies**: visualize with boxplots and histograms
- [ ] **Data Quality Summary**: actionable recommendations for preprocessing

---

## 1️⃣ Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 2️⃣ Load Dataset

In [ ]:
# TODO: Ensure data/house_prices.csv exists (download from Kaggle if needed)
df = pd.read_csv('../data/house_prices.csv')

print(f"Dataset loaded successfully.")
print(f"Shape: {df.shape}")

## 3️⃣ Dataset Shape & Size

In [ ]:
# TODO: Inspect basic dataset structure
print("=" * 60)
print("DATASET SHAPE & SIZE")
print("=" * 60)

print(f"Number of rows: {df.shape[0]}")
print(f"Number of columns: {df.shape[1]}")
print(f"\nMemory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"\nFirst few rows:")
print(df.head())

## 4️⃣ Column Names & Data Types

In [ ]:
# TODO: Verify column names and data types; identify categoricals vs numerics
print("=" * 60)
print("COLUMNS & DATA TYPES")
print("=" * 60)

info_df = pd.DataFrame({
    'Column': df.columns,
    'Data Type': df.dtypes.values,
    'Non-Null Count': df.count().values,
    'Null Count': df.isnull().sum().values,
})
print(info_df.to_string(index=False))

print(f"\nNumeric columns: {df.select_dtypes(include=[np.number]).columns.tolist()}")
print(f"Categorical columns: {df.select_dtypes(include=['object']).columns.tolist()}")

## 5️⃣ Missing Values

In [ ]:
# TODO: Identify missing values per column and percentage
print("=" * 60)
print("MISSING VALUES")
print("=" * 60)

missing_df = pd.DataFrame({
    'Column': df.columns,
    'Missing Count': df.isnull().sum().values,
    'Missing %': (df.isnull().sum() / len(df) * 100).values,
})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing %', ascending=False)

if missing_df.empty:
    print("✓ No missing values detected.")
else:
    print(missing_df.to_string(index=False))

## 6️⃣ Duplicate Rows

In [ ]:
# TODO: Identify exact duplicates and subset duplicates
print("=" * 60)
print("DUPLICATE ROWS")
print("=" * 60)

total_duplicates = df.duplicated().sum()
print(f"Total exact duplicates: {total_duplicates}")

if total_duplicates > 0:
    print(f"\nExample duplicated rows:")
    print(df[df.duplicated(keep=False)].head(10))

## 7️⃣ Target Variable (Price)

In [ ]:
# TODO: Analyze target variable distribution and identify missing or anomalous values
print("=" * 60)
print("TARGET VARIABLE: PRICE")
print("=" * 60)

# TODO: Replace 'price' with the actual column name if different
target_col = 'price'  # ADJUST IF NEEDED

if target_col in df.columns:
    print(f"\nData type: {df[target_col].dtype}")
    print(f"Missing values: {df[target_col].isnull().sum()}")
    print(f"\nDescriptive statistics:")
    print(df[target_col].describe())
else:
    print(f"⚠️  Column '{target_col}' not found. Available columns: {df.columns.tolist()}")

## 8️⃣ Numerical Features Distribution

In [ ]:
# TODO: Compute descriptive stats for all numeric columns
print("=" * 60)
print("NUMERICAL FEATURES STATISTICS")
print("=" * 60)

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

print(f"\nNumeric columns ({len(numeric_cols)}):")
print(df[numeric_cols].describe().T)

## 9️⃣ Categorical Features Distribution

In [ ]:
# TODO: Analyze unique values and distributions for categorical features
print("=" * 60)
print("CATEGORICAL FEATURES")
print("=" * 60)

categorical_cols = df.select_dtypes(include=['object']).columns.tolist()

for col in categorical_cols:
    print(f"\n{col}:")
    print(f"  Unique values: {df[col].nunique()}")
    print(f"  Top 5 values:")
    print(df[col].value_counts().head())

## 🔟 Outlier Detection (IQR Method)

In [ ]:
# TODO: Identify outliers using the Interquartile Range (IQR) method
print("=" * 60)
print("OUTLIERS (IQR METHOD)")
print("=" * 60)

for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    
    if len(outliers) > 0:
        print(f"\n{col}:")
        print(f"  Lower bound: {lower_bound:.2f}, Upper bound: {upper_bound:.2f}")
        print(f"  Outlier count: {len(outliers)} ({len(outliers)/len(df)*100:.2f}%)")

## 1️⃣1️⃣ Visualizations: Target Variable

In [ ]:
# TODO: Plot target variable distribution
if target_col in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Histogram
    axes[0].hist(df[target_col], bins=50, edgecolor='black', alpha=0.7)
    axes[0].set_xlabel(target_col.capitalize())
    axes[0].set_ylabel('Frequency')
    axes[0].set_title(f'Distribution of {target_col.capitalize()}')
    axes[0].grid(True, alpha=0.3)
    
    # Boxplot
    axes[1].boxplot(df[target_col])
    axes[1].set_ylabel(target_col.capitalize())
    axes[1].set_title(f'Boxplot of {target_col.capitalize()}')
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## 1️⃣2️⃣ Visualizations: Numerical Features

In [ ]:
# TODO: Plot distributions and boxplots for numerical features
numeric_cols_plot = [col for col in numeric_cols if col != target_col][:8]  # Limit to first 8

fig, axes = plt.subplots(len(numeric_cols_plot), 2, figsize=(14, 4*len(numeric_cols_plot)))

if len(numeric_cols_plot) == 1:
    axes = [axes]

for idx, col in enumerate(numeric_cols_plot):
    # Histogram
    axes[idx][0].hist(df[col].dropna(), bins=30, edgecolor='black', alpha=0.7)
    axes[idx][0].set_xlabel(col)
    axes[idx][0].set_ylabel('Frequency')
    axes[idx][0].set_title(f'Distribution of {col}')
    axes[idx][0].grid(True, alpha=0.3)
    
    # Boxplot
    axes[idx][1].boxplot(df[col].dropna())
    axes[idx][1].set_ylabel(col)
    axes[idx][1].set_title(f'Boxplot of {col}')
    axes[idx][1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 1️⃣3️⃣ Data Leakage Risk Assessment

In [ ]:
# TODO: Identify potential data leakage risks
print("=" * 60)
print("DATA LEAKAGE RISK ASSESSMENT")
print("=" * 60)

print("""
QUESTIONS TO CHECK:
1. Are there redundant or derived columns that directly encode the target?
2. Are there ID or index columns that should be excluded?
3. Are there temporal features that might cause leakage in production?
4. Are there features computed from the target variable?

TODO: Review columns and flag potential leakage sources below:
""")

# Placeholder for manual inspection
print(f"\nDataset columns to review:\n{df.columns.tolist()}")

## 1️⃣4️⃣ Data Quality Summary & Recommendations

In [ ]:
# TODO: Compile actionable data quality recommendations
print("=" * 60)
print("DATA QUALITY SUMMARY & RECOMMENDATIONS")
print("=" * 60)

print("""
AUDIT CHECKLIST STATUS:

1. ✓ Dataset Shape & Size: INSPECTED
2. ✓ Column Names & Data Types: INSPECTED
3. ✓ Missing Values: INSPECTED
4. ✓ Duplicate Rows: INSPECTED
5. ✓ Target Variable (Price): INSPECTED
6. ✓ Feature Distributions: INSPECTED
7. ✓ Categorical Features: INSPECTED
8. ✓ Numerical Features: INSPECTED
9. ✓ Outliers & Anomalies: INSPECTED
10. ⏳ Data Leakage Risks: MANUAL REVIEW NEEDED

NEXT STEPS (Preprocessing Phase - C2):
- TODO: Handle missing values (imputation strategy)
- TODO: Remove or address duplicates
- TODO: Handle outliers (scaling, transformation, removal)
- TODO: Encode categorical variables
- TODO: Feature engineering and selection
- TODO: Address any leakage risks identified
""")

print("\n" + "=" * 60)
print("Data Audit Complete!")
print("=" * 60)